# Обучение LSTM-модели на хронологической выборке

## Импортирование библиотек и загрузка данных

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import random

from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler

from pathlib import Path

import joblib

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

OUTPUT_DIR = Path("../data/processed")

X_train = np.load(OUTPUT_DIR / "X_train_r2.npy")
y_train = np.load(OUTPUT_DIR / "y_train_r2.npy")
X_test = np.load(OUTPUT_DIR / "X_test_r2.npy")
y_test = np.load(OUTPUT_DIR / "y_test_r2.npy")
scaler = joblib.load(OUTPUT_DIR / "scaler_r2.joblib")

WINDOW_SIZE = 12
HIDDEN_SIZE = 50

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

print(f"Размерность X_train: {X_train_t.shape}")
print(f"Размерность X_test:  {X_test_t.shape}")

## Рекурсивное прогнозирование

### Архитектура нейросети

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=50, output_size=1):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :]) 
        return out

class EarlyStopping:
    def __init__(self, patience=10, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        self.best_model_state = None

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.best_model_state = model.state_dict()
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.best_model_state = model.state_dict()
            self.counter = 0

### Данные до марта 2022

#### Настройка и обучение

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LSTMModel(input_size=1, hidden_size=HIDDEN_SIZE).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
early_stopping = EarlyStopping(patience=15)

val_size = int(len(X_train_t) * 0.2)
train_size = len(X_train_t) - val_size

X_tr, X_val = X_train_t[:train_size], X_train_t[train_size:]
y_tr, y_val = y_train_t[:train_size], y_train_t[train_size:]

train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=16, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=16, shuffle=False)

epochs = 1000
train_losses, val_losses = [], []

for epoch in range(epochs):
    model.train()
    epoch_train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        epoch_train_loss += loss.item() * X_batch.size(0)
    
    epoch_train_loss /= len(train_loader.dataset)
    train_losses.append(epoch_train_loss)

    model.eval()
    epoch_val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            epoch_val_loss += loss.item() * X_batch.size(0)
    
    epoch_val_loss /= len(val_loader.dataset)
    val_losses.append(epoch_val_loss)

    early_stopping(epoch_val_loss, model)
    if early_stopping.early_stop:
        print(f"Ранняя остановка на эпохе {epoch}")
        break

model.load_state_dict(early_stopping.best_model_state)
print("Обучение завершено.")

#### Рекурсивный прогноз на апрель - декабрь 2022

In [ ]:
df_full = pd.read_csv(OUTPUT_DIR / "cpi_monthly.csv", parse_dates=["Date"])
last_real_cpi = df_full[df_full['Date'] <= '2022-03-01']['cpi_mom'].values
scaled_last_window = scaler.transform(last_real_cpi.reshape(-1, 1))

current_window = scaled_last_window[-WINDOW_SIZE:].reshape(1, WINDOW_SIZE, 1)
current_window_t = torch.tensor(current_window, dtype=torch.float32).to(device)
predictions_scaled = []

model.eval()
with torch.no_grad():
    for step in range(9):
        next_pred_scaled = model(current_window_t).cpu().numpy()[0, 0]
        predictions_scaled.append(next_pred_scaled)
        
        next_pred_tensor = torch.tensor([[[next_pred_scaled]]], dtype=torch.float32).to(device)
        current_window_t = torch.cat((current_window_t[:, 1:, :], next_pred_tensor), dim=1)

predictions_real = scaler.inverse_transform(np.array(predictions_scaled).reshape(-1, 1))[:, 0]

months = ['Апрель', 'Май', 'Июнь', 'Июль', 'Август', 'Сентябрь', 'Октябрь', 'Ноябрь', 'Декабрь']
for month, val in zip(months, predictions_real):
    print(f"{month} 2022: {val:.2f}%")

#### Расчёт годовой инфляции за 2022 год

In [ ]:
actual_q1_2022 = df_full[(df_full['Date'] >= '2022-01-01') & (df_full['Date'] <= '2022-03-01')]['cpi_mom'].values

full_year_2022_mom = np.concatenate([actual_q1_2022, predictions_real])
annual_cpi = np.prod(full_year_2022_mom / 100)
annual_inflation = (annual_cpi - 1) * 100

print(f"Прогноз годовой инфляции на 2022 год: {annual_inflation:.2f}%")
print(f"В статье: 20.00%")
print(f"Реальная инфляция: 11.94%")

#### Визуализация прогноза

In [ ]:
forecast_dates = pd.date_range(start='2022-04-01', periods=9, freq='MS')

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_full['Date'].tail(36), df_full['cpi_mom'].tail(36), label='Фактические значения', color='steelblue')
ax.plot(forecast_dates, predictions_real, label='Прогноз LSTM', color='green', marker='o', linestyle='--')

ax.set_title('Прогноз ИПЦ на 9 месяцев вперед')
ax.set_xlabel('Дата')
ax.set_ylabel('cpi_mom, %')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

### Данные до декабря 2021

#### Формирование слепого рекурсивного прогноза на двенадцать месяцев 2022 года

In [ ]:
df_full = pd.read_csv(OUTPUT_DIR / "cpi_monthly.csv", parse_dates=["Date"])

last_2021_cpi = df_full[df_full['Date'] <= '2021-12-01']['cpi_mom'].values
scaled_2021_window = scaler.transform(last_2021_cpi.reshape(-1, 1))

current_window_2021 = scaled_2021_window[-WINDOW_SIZE:].reshape(1, WINDOW_SIZE, 1)
current_window_2021_t = torch.tensor(current_window_2021, dtype=torch.float32).to(device)

predictions_blind_scaled = []

model.eval()
with torch.no_grad():
    for step in range(12):
        next_pred_scaled = model(current_window_2021_t).cpu().numpy()[0, 0]
        predictions_blind_scaled.append(next_pred_scaled)
        
        next_pred_tensor = torch.tensor([[[next_pred_scaled]]], dtype=torch.float32).to(device)
        current_window_2021_t = torch.cat((current_window_2021_t[:, 1:, :], next_pred_tensor), dim=1)

predictions_blind_real = scaler.inverse_transform(np.array(predictions_blind_scaled).reshape(-1, 1))[:, 0]

months_full = ['Январь', 'Февраль', 'Март', 'Апрель', 'Май', 'Июнь', 'Июль', 'Август', 'Сентябрь', 'Октябрь', 'Ноябрь', 'Декабрь']
print("Слепой прогноз на 2022 год без марта:")
for month, val in zip(months_full, predictions_blind_real):
    print(f"{month} 2022: {val:.2f}%")

#### Вычисление годовой инфляции на основе слепого рекурсивного прогноза

In [ ]:
annual_cpi_blind = np.prod(predictions_blind_real / 100)
annual_inflation_blind = (annual_cpi_blind - 1) * 100

print(f"Слепой прогноз годовой инфляции на 2022 год: {annual_inflation_blind:.2f}%")

#### Графическое представление результатов слепого рекурсивного прогнозирования 

In [ ]:
forecast_dates_blind = pd.date_range(start='2022-01-01', periods=12, freq='MS')

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_full['Date'].tail(24), df_full['cpi_mom'].tail(24), label='Фактические значения', color='steelblue')
ax.plot(forecast_dates_blind, predictions_blind_real, label='Слепой прогноз LSTM', color='orange', marker='o', linestyle='--')

ax.set_title('Слепой прогноз ИПЦ на 2022 год')
ax.set_xlabel('Дата')
ax.set_ylabel('cpi_mom, %')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

## Прямое прогнозирование

### Архитектура прямой модели

In [ ]:
class LSTMDirect(nn.Module):
    def __init__(self, input_size=1, hidden_size=50, output_size=9):
        super(LSTMDirect, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :]) 
        return out

In [ ]:
FORECAST_HORIZON = 9

X_direct_train = np.load(OUTPUT_DIR / "X_direct_train.npy")
Y_direct_train = np.load(OUTPUT_DIR / "Y_direct_train.npy")
X_direct_test = np.load(OUTPUT_DIR / "X_direct_test.npy")

X_direct_train_t = torch.tensor(X_direct_train, dtype=torch.float32)
Y_direct_train_t = torch.tensor(Y_direct_train, dtype=torch.float32)
X_direct_test_t = torch.tensor(X_direct_test, dtype=torch.float32)

print(f"X_direct_train: {X_direct_train_t.shape}")
print(f"Y_direct_train: {Y_direct_train_t.shape}")
print(f"X_direct_test:  {X_direct_test_t.shape}")

### Данные до марта 2022

#### Процесс обучения модели прямого многошагового прогнозирования

In [ ]:
model_direct = LSTMDirect(input_size=1, hidden_size=HIDDEN_SIZE, output_size=FORECAST_HORIZON).to(device)
criterion_direct = nn.MSELoss()
optimizer_direct = optim.Adam(model_direct.parameters(), lr=0.001)
early_stopping_direct = EarlyStopping(patience=15)

val_size_d = int(len(X_direct_train_t) * 0.2)
train_size_d = len(X_direct_train_t) - val_size_d

X_tr_d, X_val_d = X_direct_train_t[:train_size_d], X_direct_train_t[train_size_d:]
Y_tr_d, Y_val_d = Y_direct_train_t[:train_size_d], Y_direct_train_t[train_size_d:]

train_loader_d = DataLoader(TensorDataset(X_tr_d, Y_tr_d), batch_size=16, shuffle=True)
val_loader_d = DataLoader(TensorDataset(X_val_d, Y_val_d), batch_size=16, shuffle=False)

epochs = 1000

for epoch in range(epochs):
    model_direct.train()
    for X_batch, Y_batch in train_loader_d:
        X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)
        optimizer_direct.zero_grad()
        Y_pred = model_direct(X_batch)
        loss = criterion_direct(Y_pred, Y_batch)
        loss.backward()
        optimizer_direct.step()

    model_direct.eval()
    epoch_val_loss = 0
    with torch.no_grad():
        for X_batch, Y_batch in val_loader_d:
            X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)
            Y_pred = model_direct(X_batch)
            loss = criterion_direct(Y_pred, Y_batch)
            epoch_val_loss += loss.item() * X_batch.size(0)
    
    epoch_val_loss /= len(val_loader_d.dataset)
    early_stopping_direct(epoch_val_loss, model_direct)
    if early_stopping_direct.early_stop:
        print(f"Ранняя остановка на эпохе {epoch}")
        break

model_direct.load_state_dict(early_stopping_direct.best_model_state)
print("Обучение прямой модели завершено.")

#### Генерация прогноза ИПЦ на девять месяцев прямым методом

In [ ]:
model_direct.eval()
with torch.no_grad():
    pred_direct_scaled = model_direct(X_direct_test_t.to(device)).cpu().numpy().flatten()

pred_direct_real = scaler.inverse_transform(pred_direct_scaled.reshape(-1, 1))[:, 0]

months = ['Апрель', 'Май', 'Июнь', 'Июль', 'Август', 'Сентябрь', 'Октябрь', 'Ноябрь', 'Декабрь']
print("Прямой прогноз на 2022 год:")
for month, val in zip(months, pred_direct_real):
    print(f"{month} 2022: {val:.2f}%")

#### Определение годовой инфляции на основе прямого прогноза

In [ ]:
actual_q1_2022 = df_full[(df_full['Date'] >= '2022-01-01') & (df_full['Date'] <= '2022-03-01')]['cpi_mom'].values

full_year_2022_direct = np.concatenate([actual_q1_2022, pred_direct_real])
annual_cpi_direct = np.prod(full_year_2022_direct / 100)
annual_inflation_direct = (annual_cpi_direct - 1) * 100

print(f"Годовая инфляция по прямому проходу: {annual_inflation_direct:.2f}%")
print(f"Реальная инфляция: 11.94%")

#### Визуальное сопоставление фактических значений и прямого прогноза

In [ ]:
forecast_dates = pd.date_range(start='2022-04-01', periods=9, freq='MS')

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_full['Date'].tail(24), df_full['cpi_mom'].tail(24), label='Фактические значения', color='steelblue')
ax.plot(forecast_dates, pred_direct_real, label='Прямой прогноз LSTM', color='purple', marker='s', linestyle='-.')

ax.set_title('Прямой прогноз ИПЦ на 9 месяцев вперед')
ax.set_xlabel('Дата')
ax.set_ylabel('cpi_mom, %')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

### Данные до декабря 2021

#### Обучение модели прямого прогнозирования с горизонтом двенадцать месяцев

In [ ]:
FORECAST_HORIZON_12 = 12

train_raw_12 = df_full[df_full['Date'] < '2022-01-01'].copy()
train_scaled_12 = scaler.transform(train_raw_12['cpi_mom'].values.reshape(-1, 1))

X_d12, Y_d12 = [], []
for i in range(WINDOW_SIZE, len(train_scaled_12) - FORECAST_HORIZON_12 + 1):
    X_d12.append(train_scaled_12[i-WINDOW_SIZE:i, 0])
    Y_d12.append(train_scaled_12[i:i+FORECAST_HORIZON_12, 0])

X_d12 = np.reshape(np.array(X_d12), (len(X_d12), WINDOW_SIZE, 1))
Y_d12 = np.array(Y_d12)

X_d12_t = torch.tensor(X_d12, dtype=torch.float32)
Y_d12_t = torch.tensor(Y_d12, dtype=torch.float32)

model_d12 = LSTMDirect(input_size=1, hidden_size=HIDDEN_SIZE, output_size=FORECAST_HORIZON_12).to(device)
criterion_d12 = nn.MSELoss()
optimizer_d12 = optim.Adam(model_d12.parameters(), lr=0.001)
early_stopping_d12 = EarlyStopping(patience=15)

val_size_d12 = int(len(X_d12_t) * 0.2)
train_size_d12 = len(X_d12_t) - val_size_d12

X_tr_d12, X_val_d12 = X_d12_t[:train_size_d12], X_d12_t[train_size_d12:]
Y_tr_d12, Y_val_d12 = Y_d12_t[:train_size_d12], Y_d12_t[train_size_d12:]

train_loader_d12 = DataLoader(TensorDataset(X_tr_d12, Y_tr_d12), batch_size=16, shuffle=True)
val_loader_d12 = DataLoader(TensorDataset(X_val_d12, Y_val_d12), batch_size=16, shuffle=False)

for epoch in range(1000):
    model_d12.train()
    for X_batch, Y_batch in train_loader_d12:
        X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)
        optimizer_d12.zero_grad()
        Y_pred = model_d12(X_batch)
        loss = criterion_d12(Y_pred, Y_batch)
        loss.backward()
        optimizer_d12.step()

    model_d12.eval()
    epoch_val_loss = 0
    with torch.no_grad():
        for X_batch, Y_batch in val_loader_d12:
            X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)
            Y_pred = model_d12(X_batch)
            loss = criterion_d12(Y_pred, Y_batch)
            epoch_val_loss += loss.item() * X_batch.size(0)
    
    epoch_val_loss /= len(val_loader_d12.dataset)
    early_stopping_d12(epoch_val_loss, model_d12)
    if early_stopping_d12.early_stop:
        print(f"Ранняя остановка на эпохе {epoch}")
        break

model_d12.load_state_dict(early_stopping_d12.best_model_state)
print("Обучение прямой модели на 12 шагов завершено.")

#### Формирование слепого прямого прогноза на двенадцать месяцев

In [ ]:
model_d12.eval()
with torch.no_grad():
    pred_d12_scaled = model_d12(current_window_2021_t).cpu().numpy().flatten()

pred_d12_real = scaler.inverse_transform(pred_d12_scaled.reshape(-1, 1))[:, 0]

print("Прямой слепой прогноз на 12 месяцев 2022 года:")
for month, val in zip(months_full, pred_d12_real):
    print(f"{month} 2022: {val:.2f}%")

#### Расчёт годовой инфляции по результатам слепого прямого прогноза

In [ ]:
annual_cpi_d12 = np.prod(pred_d12_real / 100)
annual_inflation_d12 = (annual_cpi_d12 - 1) * 100

print(f"Годовая инфляция: {annual_inflation_d12:.2f}%")

## Seq2Seg-прогнозирование с механизмом внимания

### Определение архитектуры кодировщика

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_size=1, hidden_size=50):
        super(Encoder, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
    
    def forward(self, x):
        outputs, (hidden, cell) = self.lstm(x)
        return outputs, hidden, cell

### Определение механизма внимания

In [ ]:
class Attention(nn.Module):
    def __init__(self, hidden_size=50):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_size * 2, hidden_size)
        self.v = nn.Linear(hidden_size, 1, bias=False)
    
    def forward(self, hidden, encoder_outputs):
        batch_size = encoder_outputs.size(0)
        seq_len = encoder_outputs.size(1)
        
        hidden = hidden.unsqueeze(1).repeat(1, seq_len, 1)
        
        energy = torch.tanh(self.attention(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)
        
        return torch.softmax(attention, dim=1)

### Определение архитектуры декодировщика

In [ ]:
class Decoder(nn.Module):
    def __init__(self, input_size=1, hidden_size=50, output_size=1):
        super(Decoder, self).__init__()
        self.lstm = nn.LSTM(input_size + hidden_size, hidden_size, batch_first=True)
        self.attention = Attention(hidden_size)
        self.fc = nn.Linear(hidden_size * 2, output_size)
    
    def forward(self, input, hidden, cell, encoder_outputs):
        input = input.unsqueeze(1)
        
        attention_weights = self.attention(hidden[-1], encoder_outputs)
        attention_weights = attention_weights.unsqueeze(1)
        
        attended = torch.bmm(attention_weights, encoder_outputs)
        
        lstm_input = torch.cat((input, attended), dim=2)
        
        output, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))
        
        prediction = self.fc(torch.cat((output.squeeze(1), attended.squeeze(1)), dim=1))
        
        return prediction, hidden, cell, attention_weights.squeeze(1)

### Определение модели seq2seq

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        self.forecast_horizon = FORECAST_HORIZON
    
    def forward(self, encoder_input, decoder_input, teacher_forcing_ratio=0.5):
        encoder_outputs, hidden, cell = self.encoder(encoder_input)
        
        input = decoder_input[:, -1]
        
        outputs = []
        attention_weights_all = []
        
        for t in range(self.forecast_horizon):
            output, hidden, cell, attention_weights = self.decoder(input, hidden, cell, encoder_outputs)
            outputs.append(output)
            attention_weights_all.append(attention_weights)
            
            teacher_force = random.random() < teacher_forcing_ratio
            
            if teacher_force:
                input = decoder_input[:, t]
            else:
                input = output
        
        outputs = torch.cat(outputs, dim=1)
        attention_weights_all = torch.stack(attention_weights_all, dim=1)
        
        return outputs, attention_weights_all

### Данные до декабря 2021

#### Подготовка последовательностей для архитектуры encoder-decoder

In [ ]:
train_raw = df_full[df_full['Date'] < '2022-01-01'].copy()

scaler_s2s = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler_s2s.fit_transform(train_raw['cpi_mom'].values.reshape(-1, 1))

X_encoder_train, X_decoder_train, Y_decoder_train = [], [], []
for i in range(WINDOW_SIZE, len(train_scaled) - FORECAST_HORIZON + 1):
    X_encoder_train.append(train_scaled[i-WINDOW_SIZE:i, 0])
    X_decoder_train.append(train_scaled[i-WINDOW_SIZE:i, 0])
    Y_decoder_train.append(train_scaled[i:i+FORECAST_HORIZON, 0])

X_encoder_train = np.array(X_encoder_train).reshape(-1, WINDOW_SIZE, 1)
X_decoder_train = np.array(X_decoder_train).reshape(-1, WINDOW_SIZE, 1)
Y_decoder_train = np.array(Y_decoder_train)

last_window = train_scaled[-WINDOW_SIZE:].reshape(1, WINDOW_SIZE, 1)
X_encoder_test = last_window
X_decoder_test = last_window

X_encoder_train_t = torch.tensor(X_encoder_train, dtype=torch.float32)
X_decoder_train_t = torch.tensor(X_decoder_train, dtype=torch.float32)
Y_decoder_train_t = torch.tensor(Y_decoder_train, dtype=torch.float32)
X_encoder_test_t = torch.tensor(X_encoder_test, dtype=torch.float32)
X_decoder_test_t = torch.tensor(X_decoder_test, dtype=torch.float32)

print(f"X_encoder_train: {X_encoder_train_t.shape}")
print(f"X_decoder_train: {X_decoder_train_t.shape}")
print(f"Y_decoder_train: {Y_decoder_train_t.shape}")

#### Обучение модели с ранней остановкой

In [ ]:
encoder = Encoder(input_size=1, hidden_size=HIDDEN_SIZE).to(device)
decoder = Decoder(input_size=1, hidden_size=HIDDEN_SIZE, output_size=1).to(device)
model_s2s = Seq2Seq(encoder, decoder, device).to(device)

criterion_s2s = nn.MSELoss()
optimizer_s2s = optim.Adam(model_s2s.parameters(), lr=0.001)

early_stopping_s2s = EarlyStopping(patience=15)

val_size = int(len(X_encoder_train_t) * 0.2)
train_size = len(X_encoder_train_t) - val_size

X_enc_tr = X_encoder_train_t[:train_size].to(device)
X_dec_tr = X_decoder_train_t[:train_size].to(device)
Y_dec_tr = Y_decoder_train_t[:train_size].to(device)

X_enc_val = X_encoder_train_t[train_size:].to(device)
X_dec_val = X_decoder_train_t[train_size:].to(device)
Y_dec_val = Y_decoder_train_t[train_size:].to(device)

train_dataset = TensorDataset(X_enc_tr, X_dec_tr, Y_dec_tr)
val_dataset = TensorDataset(X_enc_val, X_dec_val, Y_dec_val)

train_loader_s2s = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader_s2s = DataLoader(val_dataset, batch_size=16, shuffle=False)

epochs = 1000
train_losses_s2s, val_losses_s2s = [], []

for epoch in range(epochs):
    model_s2s.train()
    epoch_train_loss = 0
    
    for X_enc_batch, X_dec_batch, Y_dec_batch in train_loader_s2s:
        optimizer_s2s.zero_grad()
        
        Y_pred, _ = model_s2s(X_enc_batch, X_dec_batch, teacher_forcing_ratio=0.5)
        loss = criterion_s2s(Y_pred, Y_dec_batch)
        
        loss.backward()
        optimizer_s2s.step()
        
        epoch_train_loss += loss.item() * X_enc_batch.size(0)
    
    epoch_train_loss /= len(train_loader_s2s.dataset)
    train_losses_s2s.append(epoch_train_loss)
    
    model_s2s.eval()
    epoch_val_loss = 0
    
    with torch.no_grad():
        for X_enc_batch, X_dec_batch, Y_dec_batch in val_loader_s2s:
            Y_pred, _ = model_s2s(X_enc_batch, X_dec_batch, teacher_forcing_ratio=0.0)
            loss = criterion_s2s(Y_pred, Y_dec_batch)
            epoch_val_loss += loss.item() * X_enc_batch.size(0)
    
    epoch_val_loss /= len(val_loader_s2s.dataset)
    val_losses_s2s.append(epoch_val_loss)
    
    early_stopping_s2s(epoch_val_loss, model_s2s)
    
    if early_stopping_s2s.early_stop:
        print(f"Ранняя остановка на эпохе {epoch}")
        break

model_s2s.load_state_dict(early_stopping_s2s.best_model_state)
print("Обучение модели Seq2Seq завершено.")

plt.figure(figsize=(10, 5))
plt.plot(train_losses_s2s, label='Train Loss', color='steelblue')
plt.plot(val_losses_s2s, label='Validation Loss', color='orange')
plt.title('Кривые обучения Seq2Seq с Attention')
plt.xlabel('Эпоха')
plt.ylabel('MSE')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

#### Формирование прогноза на 2022 год

In [ ]:
model_s2s.eval()

X_encoder_test_t = X_encoder_test_t.to(device)
X_decoder_test_t = X_decoder_test_t.to(device)

with torch.no_grad():
    pred_s2s_scaled, attention_weights = model_s2s(X_encoder_test_t, X_decoder_test_t, teacher_forcing_ratio=0.0)
    pred_s2s_scaled = pred_s2s_scaled.cpu().numpy().flatten()
    attention_weights = attention_weights.cpu().numpy()

pred_s2s_real = scaler_s2s.inverse_transform(pred_s2s_scaled.reshape(-1, 1))[:, 0]

months = ['Апрель', 'Май', 'Июнь', 'Июль', 'Август', 'Сентябрь', 'Октябрь', 'Ноябрь', 'Декабрь']
print("Прогноз Seq2Seq с механизмом внимания на 2022 год:")
for month, val in zip(months, pred_s2s_real):
    print(f"{month} 2022: {val:.2f}%")

forecast_dates = pd.date_range(start='2022-04-01', periods=9, freq='MS')
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_full['Date'].tail(24), df_full['cpi_mom'].tail(24), label='Фактические значения', color='steelblue')
ax.plot(forecast_dates, pred_s2s_real, label='Прогноз Seq2Seq', color='darkred', marker='D', linestyle='-.')
ax.set_title('Прогноз ИПЦ на 9 месяцев вперед')
ax.set_xlabel('Дата')
ax.set_ylabel('cpi_mom, %')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

#### Визуализация весов механизма внимания

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

attention_mean = attention_weights[0].mean(axis=0)

input_months = ['Янв-21', 'Фев-21', 'Мар-21', 'Апр-21', 'Май-21', 'Июн-21', 
                'Июл-21', 'Авг-21', 'Сен-21', 'Окт-21', 'Ноя-21', 'Дек-21']

im = ax.imshow(attention_weights[0].T, cmap='YlOrRd', aspect='auto')

ax.set_yticks(range(len(input_months)))
ax.set_yticklabels(input_months)

ax.set_xticks([])

ax.set_ylabel('Входная последовательность, окно 12 месяцев')
ax.set_title('Влияние каждого входного месяца на прогноз')

plt.colorbar(im, ax=ax, label='Вес внимания')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(len(input_months)), attention_mean, color='darkred', alpha=0.7)
ax.set_xticks(range(len(input_months)))
ax.set_xticklabels(input_months, rotation=45, ha='right')
ax.set_xlabel('Входной месяц')
ax.set_ylabel('Средний вес внимания')
ax.set_title('Средняя важность каждого входного месяца для прогноза')
ax.grid(True, linestyle='--', alpha=0.6, axis='y')
plt.tight_layout()
plt.show()

#### Расчёт годовой инфляции по прогнозу

In [ ]:
actual_q1_2022 = df_full[(df_full['Date'] >= '2022-01-01') & (df_full['Date'] <= '2022-03-01')]['cpi_mom'].values
full_year_2022_s2s = np.concatenate([actual_q1_2022, pred_s2s_real])

annual_cpi_s2s = np.prod(full_year_2022_s2s / 100)
annual_inflation_s2s = (annual_cpi_s2s - 1) * 100

print(f"Прогноз годовой инфляции на 2022 год: {annual_inflation_s2s:.2f}%")

### Данные до марта 2022

#### Подготовка последовательностей для архитектуры encoder-decoder

In [ ]:
train_raw = df_full[df_full['Date'] <= '2022-03-01'].copy()

scaler_s2s = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler_s2s.fit_transform(train_raw['cpi_mom'].values.reshape(-1, 1))

X_encoder_train, X_decoder_train, Y_decoder_train = [], [], []
for i in range(WINDOW_SIZE, len(train_scaled) - FORECAST_HORIZON + 1):
    X_encoder_train.append(train_scaled[i-WINDOW_SIZE:i, 0])
    X_decoder_train.append(train_scaled[i-WINDOW_SIZE:i, 0])
    Y_decoder_train.append(train_scaled[i:i+FORECAST_HORIZON, 0])

X_encoder_train = np.array(X_encoder_train).reshape(-1, WINDOW_SIZE, 1)
X_decoder_train = np.array(X_decoder_train).reshape(-1, WINDOW_SIZE, 1)
Y_decoder_train = np.array(Y_decoder_train)

last_window = train_scaled[-WINDOW_SIZE:].reshape(1, WINDOW_SIZE, 1)
X_encoder_test = last_window
X_decoder_test = last_window

X_encoder_train_t = torch.tensor(X_encoder_train, dtype=torch.float32)
X_decoder_train_t = torch.tensor(X_decoder_train, dtype=torch.float32)
Y_decoder_train_t = torch.tensor(Y_decoder_train, dtype=torch.float32)
X_encoder_test_t = torch.tensor(X_encoder_test, dtype=torch.float32)
X_decoder_test_t = torch.tensor(X_decoder_test, dtype=torch.float32)

print(f"X_encoder_train: {X_encoder_train_t.shape}")
print(f"X_decoder_train: {X_decoder_train_t.shape}")
print(f"Y_decoder_train: {Y_decoder_train_t.shape}")
print(f"Данные для обучения: с {train_raw['Date'].min().date()} по {train_raw['Date'].max().date()}")

#### Обучение модели с ранней остановкой 

In [ ]:
encoder = Encoder(input_size=1, hidden_size=HIDDEN_SIZE).to(device)
decoder = Decoder(input_size=1, hidden_size=HIDDEN_SIZE, output_size=1).to(device)
model_s2s = Seq2Seq(encoder, decoder, device).to(device)

criterion_s2s = nn.MSELoss()
optimizer_s2s = optim.Adam(model_s2s.parameters(), lr=0.001)

early_stopping_s2s = EarlyStopping(patience=15)

val_size = int(len(X_encoder_train_t) * 0.2)
train_size = len(X_encoder_train_t) - val_size

X_enc_tr = X_encoder_train_t[:train_size].to(device)
X_dec_tr = X_decoder_train_t[:train_size].to(device)
Y_dec_tr = Y_decoder_train_t[:train_size].to(device)

X_enc_val = X_encoder_train_t[train_size:].to(device)
X_dec_val = X_decoder_train_t[train_size:].to(device)
Y_dec_val = Y_decoder_train_t[train_size:].to(device)

train_dataset = TensorDataset(X_enc_tr, X_dec_tr, Y_dec_tr)
val_dataset = TensorDataset(X_enc_val, X_dec_val, Y_dec_val)

train_loader_s2s = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader_s2s = DataLoader(val_dataset, batch_size=16, shuffle=False)

epochs = 1000
train_losses_s2s, val_losses_s2s = [], []

for epoch in range(epochs):
    model_s2s.train()
    epoch_train_loss = 0
    
    for X_enc_batch, X_dec_batch, Y_dec_batch in train_loader_s2s:
        optimizer_s2s.zero_grad()
        
        Y_pred, _ = model_s2s(X_enc_batch, X_dec_batch, teacher_forcing_ratio=0.5)
        loss = criterion_s2s(Y_pred, Y_dec_batch)
        
        loss.backward()
        optimizer_s2s.step()
        
        epoch_train_loss += loss.item() * X_enc_batch.size(0)
    
    epoch_train_loss /= len(train_loader_s2s.dataset)
    train_losses_s2s.append(epoch_train_loss)
    
    model_s2s.eval()
    epoch_val_loss = 0
    
    with torch.no_grad():
        for X_enc_batch, X_dec_batch, Y_dec_batch in val_loader_s2s:
            Y_pred, _ = model_s2s(X_enc_batch, X_dec_batch, teacher_forcing_ratio=0.0)
            loss = criterion_s2s(Y_pred, Y_dec_batch)
            epoch_val_loss += loss.item() * X_enc_batch.size(0)
    
    epoch_val_loss /= len(val_loader_s2s.dataset)
    val_losses_s2s.append(epoch_val_loss)
    
    early_stopping_s2s(epoch_val_loss, model_s2s)
    
    if early_stopping_s2s.early_stop:
        print(f"Ранняя остановка на эпохе {epoch}")
        break

model_s2s.load_state_dict(early_stopping_s2s.best_model_state)
print("Обучение модели Seq2Seq завершено.")

plt.figure(figsize=(10, 5))
plt.plot(train_losses_s2s, label='Train Loss', color='steelblue')
plt.plot(val_losses_s2s, label='Validation Loss', color='orange')
plt.title('Кривые обучения Seq2Seq с Attention')
plt.xlabel('Эпоха')
plt.ylabel('MSE')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

#### Формирование прогноза на период с апреля по декабрь 2022 года 

In [ ]:
model_s2s.eval()

X_encoder_test_t = X_encoder_test_t.to(device)
X_decoder_test_t = X_decoder_test_t.to(device)

with torch.no_grad():
    pred_s2s_scaled, attention_weights = model_s2s(X_encoder_test_t, X_decoder_test_t, teacher_forcing_ratio=0.0)
    pred_s2s_scaled = pred_s2s_scaled.cpu().numpy().flatten()
    attention_weights = attention_weights.cpu().numpy()

pred_s2s_real = scaler_s2s.inverse_transform(pred_s2s_scaled.reshape(-1, 1))[:, 0]

months = ['Апрель', 'Май', 'Июнь', 'Июль', 'Август', 'Сентябрь', 'Октябрь', 'Ноябрь', 'Декабрь']
print("Прогноз Seq2Seq с механизмом внимания на 2022 год:")
for month, val in zip(months, pred_s2s_real):
    print(f"{month} 2022: {val:.2f}%")

forecast_dates = pd.date_range(start='2022-04-01', periods=9, freq='MS')
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_full['Date'].tail(24), df_full['cpi_mom'].tail(24), label='Фактические значения', color='steelblue')
ax.plot(forecast_dates, pred_s2s_real, label='Прогноз Seq2Seq', color='darkred', marker='D', linestyle='-.')
ax.set_title('Прогноз ИПЦ на 9 месяцев вперед')
ax.set_xlabel('Дата')
ax.set_ylabel('cpi_mom, %')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

#### Визуализация весов механизма внимания

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

attention_mean = attention_weights[0].mean(axis=0)

input_months = ['Апр-21', 'Май-21', 'Июн-21', 'Июл-21', 'Авг-21', 'Сен-21', 
                'Окт-21', 'Ноя-21', 'Дек-21', 'Янв-22', 'Фев-22', 'Мар-22']

im = ax.imshow(attention_weights[0].T, cmap='YlOrRd', aspect='auto')

ax.set_yticks(range(len(input_months)))
ax.set_yticklabels(input_months)

ax.set_xticks([])

ax.set_ylabel('Входная последовательность, окно 12 месяцев')
ax.set_title('Влияние каждого входного месяца на прогноз')

plt.colorbar(im, ax=ax, label='Вес внимания')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(len(input_months)), attention_mean, color='darkred', alpha=0.7)
ax.set_xticks(range(len(input_months)))
ax.set_xticklabels(input_months, rotation=45, ha='right')
ax.set_xlabel('Входной месяц')
ax.set_ylabel('Средний вес внимания')
ax.set_title('Средняя важность каждого входного месяца для прогноза')
ax.grid(True, linestyle='--', alpha=0.6, axis='y')
plt.tight_layout()
plt.show()

#### Расчёт годовой инфляции по прогнозу

In [ ]:
actual_q1_2022 = df_full[(df_full['Date'] >= '2022-01-01') & (df_full['Date'] <= '2022-03-01')]['cpi_mom'].values
full_year_2022_s2s = np.concatenate([actual_q1_2022, pred_s2s_real])

annual_cpi_s2s = np.prod(full_year_2022_s2s / 100)
annual_inflation_s2s = (annual_cpi_s2s - 1) * 100

print(f"Прогноз годовой инфляции на 2022 год: {annual_inflation_s2s:.2f}%")